<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/4_Aprendizaje_no_supervisado/2_Taller_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# **Taller: Análisis de Patrones de Consumo Internacional con Apriori**

**IMPORTANTE**: Guarda una copia de este notebook en tu Google Drive o computador.

**Taller en grupos de 3**

**Nombres estudiantes:**
Marleny Cuaspud Tarapues
-
-
-

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma:“Taller_Apriori_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/qERdEpXpmx.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Plazo de entrega:**

21 de abril de 2026, máximo a las 11:59 p.m. Tenga en cuenta que luego de esa hora el formulario en forms se cierra. El Jupupyter Notebook también debe quedar subido en Github antes de esa hora.

**Instrucciones Generales:**

Completa el código en las celdas marcadas con `### TU CÓDIGO AQUÍ ###`. Puedes añadir más celdas si lo requieres.

**Caso de Estudio: Consultoría para Global Retail Inc.**

**Contexto:** Una firma multinacional de e-commerce, "Global Retail Inc.", te ha contratado como consultor de datos. La empresa opera en múltiples países y ha notado que sus ventas y la efectividad de sus campañas de marketing varían significativamente entre regiones. Su hipótesis es que los patrones de compra y las asociaciones de productos son diferentes en cada mercado.

**Tu Misión:** Analizar el historial de transacciones de la empresa para descubrir y comparar las reglas de asociación de productos para dos de sus mercados más importantes en Latinoamérica: México y Colombia. Tu objetivo final es entregar recomendaciones de negocio accionables (ej. estrategias de cross-selling, promociones personalizadas) basadas en los patrones de consumo que descubras en cada país.

**Dataset:** Encuentra mayor información en el archivo "diccionario_alimentos_retail_top30.xlsx".

## Ejercicio 1: Configuración Inicial, Carga y Exploración de Datos

1.1 Importa las librerías necesarias

In [ ]:
### TU CÓDIGO AQUÍ ###
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
import os

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from IPython.core.display import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Configuraciones de visualización
pd.options.display.max_columns = None
pd.options.display.float_format = '{:,.2f}'.format

1.2 Carga el dataset "alimentos_retail_top30.csv" que se encuentra en el repositorio del curso, carpeta "datasets". El dataframe debe llamarse "df".

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path = '/content/drive/MyDrive/Datasets/'
# Para establecer el directorio de los archivos
os.chdir(path)

In [ ]:
df = pd.read_csv("alimentos_retail_top30.csv")
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536000,94537,HARINA DE MAÍZ,5,2023-01-07 01:09:00,2.76,"17,452.00",Colombia
1,536000,87297,QUESO MUZZARELLA,2,2023-01-07 01:09:00,4.69,"17,779.00",Colombia
2,536001,94537,HARINA DE MAÍZ,4,2023-01-07 11:51:00,2.76,"14,933.00",Colombia
3,536001,87297,QUESO MUZZARELLA,3,2023-01-07 11:51:00,4.69,"14,957.00",Colombia
4,536002,26907,CAFÉ,4,2023-01-02 01:54:00,2.36,"15,202.00",Colombia
...,...,...,...,...,...,...,...,...
6894,537998,36301,TORTILLAS DE MAÍZ,4,2023-01-05 14:28:00,4.14,"13,520.00",México
6895,537999,48011,FRIJOL NEGRO,1,2023-01-07 20:26:00,1.86,"12,105.00",México
6896,537999,36355,TOMATE,-1,2023-01-07 20:26:00,4.87,"16,918.00",México
6897,537999,36301,TORTILLAS DE MAÍZ,2,2023-01-07 20:26:00,4.14,"15,425.00",México


In [44]:
df['Description'] = df['Description'].str.strip()


In [ ]:
# Debe ser (6899, 8)
print("Dimensiones del DataFrame:")
print(df.shape)

Dimensiones del DataFrame:
(6899, 8)


In [ ]:
print("\nInformación general del DataFrame:")
df.info()


Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6899 entries, 0 to 6898
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    6899 non-null   object 
 1   StockCode    6899 non-null   int64  
 2   Description  6899 non-null   object 
 3   Quantity     6899 non-null   int64  
 4   InvoiceDate  6899 non-null   object 
 5   UnitPrice    6899 non-null   float64
 6   CustomerID   6879 non-null   float64
 7   Country      6899 non-null   object 
dtypes: float64(2), int64(2), object(4)
memory usage: 431.3+ KB


1.3 Revisa si hay valores nulos en alguna columna y cuántos son

In [ ]:
### TU CÓDIGO AQUÍ ###
df.isnull().sum()

,0
InvoiceNo,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,20
Country,0


1.4 Genera las estadísticas descriptivas de las variables numéricas

In [ ]:
### TU CÓDIGO AQUÍ ###
statistics = df.describe()
statistics

,StockCode,Quantity,UnitPrice,CustomerID
count,"6,899.00","6,899.00","6,899.00","6,879.00"
mean,"55,544.94",3.00,3.42,"15,024.12"
std,"25,875.73",1.43,1.06,"1,732.95"
min,"26,907.00",-5.00,1.65,"12,000.00"
25%,"31,048.00",2.00,2.36,"13,524.00"
50%,"42,889.00",3.00,3.39,"15,041.00"
75%,"87,297.00",4.00,4.44,"16,530.50"
max,"95,931.00",5.00,4.90,"17,999.00"


1.5 Observando las salidas del ejercicio anterior, ¿qué problemas potenciales identificas en las columnas CustomerID y Quantity?

## Ejercicio 2: Limpieza y Preprocesamiento de Datos

Los datos del mundo real rara vez son perfectos. Antes de cualquier análisis, debemos "sanear" nuestro dataset. Completa el código en cada paso según las instrucciones.

Crea un nuevo dataframe llamado "df_limpio" para los siguientes puntos.

2.1 **Manejo de Valores Nulos**: Las transacciones sin un CustomerID no son útiles para nosotros, ya que no podemos agrupar las compras de un cliente específico.

In [ ]:
# TAREA: Elimina todas las filas donde 'CustomerID' es nulo.
### TU CÓDIGO AQUÍ ###
df_limpio = df.dropna(subset=['CustomerID'])
df_limpio

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536000,94537,HARINA DE MAÍZ,5,2023-01-07 01:09:00,2.76,"17,452.00",Colombia
1,536000,87297,QUESO MUZZARELLA,2,2023-01-07 01:09:00,4.69,"17,779.00",Colombia
2,536001,94537,HARINA DE MAÍZ,4,2023-01-07 11:51:00,2.76,"14,933.00",Colombia
3,536001,87297,QUESO MUZZARELLA,3,2023-01-07 11:51:00,4.69,"14,957.00",Colombia
4,536002,26907,CAFÉ,4,2023-01-02 01:54:00,2.36,"15,202.00",Colombia
...,...,...,...,...,...,...,...,...
6894,537998,36301,TORTILLAS DE MAÍZ,4,2023-01-05 14:28:00,4.14,"13,520.00",México
6895,537999,48011,FRIJOL NEGRO,1,2023-01-07 20:26:00,1.86,"12,105.00",México
6896,537999,36355,TOMATE,-1,2023-01-07 20:26:00,4.87,"16,918.00",México
6897,537999,36301,TORTILLAS DE MAÍZ,2,2023-01-07 20:26:00,4.14,"15,425.00",México


In [ ]:
# El tipo de dato de CustomerID debe ser entero
### TU CÓDIGO AQUÍ ###
df = df_limpio.astype({'CustomerID': 'int64'})
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6879 entries, 0 to 6898
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    6879 non-null   object 
 1   StockCode    6879 non-null   int64  
 2   Description  6879 non-null   object 
 3   Quantity     6879 non-null   int64  
 4   InvoiceDate  6879 non-null   object 
 5   UnitPrice    6879 non-null   float64
 6   CustomerID   6879 non-null   int64  
 7   Country      6879 non-null   object 
dtypes: float64(1), int64(3), object(4)
memory usage: 741.7+ KB


2.2 **Limpieza de Descripciones de Productos** Las descripciones pueden tener espacios en blanco al inicio o al final que podrían hacer que un mismo producto se cuente como dos diferentes.

In [ ]:
# TAREA: # Verifica cuántas descripciones únicas hay.
### TU CÓDIGO AQUÍ ###
descripciones_unicas = df_limpio['Description'].nunique()
print("Número de descripciones únicas:", descripciones_unicas)

Número de descripciones únicas: 25


In [ ]:
# TAREA: Limpia la columna 'Description' eliminando espacios extra al inicio y al final.
### TU CÓDIGO AQUÍ ###
df_limpio['Description'] = df_limpio['Description'].str.strip()

In [ ]:
# TAREA: Verifica cuántas descripciones únicas quedaron después de la limpieza.
### TU CÓDIGO AQUÍ ###
descripciones_unicas_limpias = df_limpio['Description'].nunique()
print("Número de descripciones únicas después de la limpieza:", descripciones_unicas_limpias)

Número de descripciones únicas después de la limpieza: 20


2.3 **Filtrado de Transacciones Anómalas**: Las facturas (InvoiceNo) que empiezan con 'C' indican una cancelación. Estas no son compras reales y deben ser eliminadas. Del mismo modo, las cantidades (Quantity) negativas representan devoluciones.

In [ ]:
# TAREA: Elimina las filas que correspondan a cancelaciones.
### TU CÓDIGO AQUÍ ###
df_limpio = df_limpio[~df_limpio['InvoiceNo'].str.startswith('C')]
df_limpio = df_limpio[df_limpio['Quantity'] >= 0]

In [ ]:
# TAREA: Elimina las filas con cantidades negativas.
### TU CÓDIGO AQUÍ ###
df = df_limpio = df_limpio[df_limpio['Quantity'] >= 0]

In [ ]:
# Verifiquemos las dimensiones del DataFrame después de la limpieza. Debe ser (6864, 8)
df_limpio.shape

(6864, 8)

## Ejercicio 3: Análisis Comparativo por País

Ahora que los datos están limpios, vamos a segmentarlos y a aplicar el algoritmo Apriori para encontrar los patrones de compra en México y Colombia.

**Preparación de la Cesta de Mercado (Función)**

La siguiente función toma un dataframe, lo agrupa por factura y descripción, y lo transforma en el formato de matriz binaria que necesita el algoritmo Apriori. Estudia esta función, no necesitas modificarla.

In [ ]:
def preparar_cesta(dataframe, pais):
    """Filtra por país y prepara la matriz de transacciones."""

    # Filtrar por el país de interés
    df_pais = dataframe[dataframe['Country'] == pais]

    # Crear la cesta: agrupar productos por factura
    cesta = (df_pais.groupby(['InvoiceNo', 'Description'])['Quantity']
             .sum().unstack().reset_index().fillna(0)
             .set_index('InvoiceNo'))

    # Convertir todas las cantidades positivas a 1 y todo lo demás a 0
    cesta_encoded = (cesta > 0).astype(int)

    return cesta_encoded

3.1 Análisis para México

In [50]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de México. Almacena el resultado en la variable cesta_mx.
### TU CÓDIGO AQUÍ ###
cesta_mx = preparar_cesta(df_limpio, 'México')

In [51]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536000,94537,HARINA DE MAÍZ,5,2023-01-07 01:09:00,2.76,"17,452.00",Colombia
1,536000,87297,QUESO MUZZARELLA,2,2023-01-07 01:09:00,4.69,"17,779.00",Colombia
2,536001,94537,HARINA DE MAÍZ,4,2023-01-07 11:51:00,2.76,"14,933.00",Colombia
3,536001,87297,QUESO MUZZARELLA,3,2023-01-07 11:51:00,4.69,"14,957.00",Colombia
4,536002,26907,CAFÉ,4,2023-01-02 01:54:00,2.36,"15,202.00",Colombia
...,...,...,...,...,...,...,...,...
6893,537998,48011,FRIJOL NEGRO,4,2023-01-05 14:28:00,1.86,"12,401.00",México
6894,537998,36301,TORTILLAS DE MAÍZ,4,2023-01-05 14:28:00,4.14,"13,520.00",México
6895,537999,48011,FRIJOL NEGRO,1,2023-01-07 20:26:00,1.86,"12,105.00",México
6897,537999,36301,TORTILLAS DE MAÍZ,2,2023-01-07 20:26:00,4.14,"15,425.00",México


In [ ]:
# TAREA: Aplica el algoritmo apriori para encontrar itemsets con un soporte mínimo de 2%.
# Almacena el resultado en la variable frequent_itemsets_mx.
# Muestra los 10 itemsets con el soporte más alto.
### TU CÓDIGO AQUÍ ###
frequent_itemsets_mx = apriori(cesta_mx, min_support=0.02, use_colnames=True)
frequent_itemsets_mx.sort_values('support', ascending=False).head(10)

,support,itemsets


In [ ]:
if not frequent_itemsets_mx.empty:
    rules_mx = association_rules(frequent_itemsets_mx, metric='lift', min_threshold=2)
else:
    rules_mx = pd.DataFrame(columns=['antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'])
    print("No se pudieron generar reglas de asociación para México porque no se encontraron itemsets frecuentes.")

No se pudieron generar reglas de asociación para México porque no se encontraron itemsets frecuentes.


In [ ]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
### TU CÓDIGO AQUÍ ###
resultado = rules_mx.sort_values(['lift', 'confidence'], ascending=False)[
    ['antecedents', 'consequents', 'antecedent support',
     'consequent support', 'confidence', 'lift']
].head(10)

In [48]:
df_limpio.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536000,94537,HARINA DE MAÍZ,5,2023-01-07 01:09:00,2.76,"17,452.00",Colombia
1,536000,87297,QUESO MUZZARELLA,2,2023-01-07 01:09:00,4.69,"17,779.00",Colombia
2,536001,94537,HARINA DE MAÍZ,4,2023-01-07 11:51:00,2.76,"14,933.00",Colombia
3,536001,87297,QUESO MUZZARELLA,3,2023-01-07 11:51:00,4.69,"14,957.00",Colombia
4,536002,26907,CAFÉ,4,2023-01-02 01:54:00,2.36,"15,202.00",Colombia


In [52]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536000,94537,HARINA DE MAÍZ,5,2023-01-07 01:09:00,2.76,"17,452.00",Colombia
1,536000,87297,QUESO MUZZARELLA,2,2023-01-07 01:09:00,4.69,"17,779.00",Colombia
2,536001,94537,HARINA DE MAÍZ,4,2023-01-07 11:51:00,2.76,"14,933.00",Colombia
3,536001,87297,QUESO MUZZARELLA,3,2023-01-07 11:51:00,4.69,"14,957.00",Colombia
4,536002,26907,CAFÉ,4,2023-01-02 01:54:00,2.36,"15,202.00",Colombia
...,...,...,...,...,...,...,...,...
6893,537998,48011,FRIJOL NEGRO,4,2023-01-05 14:28:00,1.86,"12,401.00",México
6894,537998,36301,TORTILLAS DE MAÍZ,4,2023-01-05 14:28:00,4.14,"13,520.00",México
6895,537999,48011,FRIJOL NEGRO,1,2023-01-07 20:26:00,1.86,"12,105.00",México
6897,537999,36301,TORTILLAS DE MAÍZ,2,2023-01-07 20:26:00,4.14,"15,425.00",México


In [54]:
mejor_reglas_co = rules_co.sort_values(['lift', 'confidence'], ascending=False)[
    ['antecedents', 'consequents', 'antecedent support',
     'consequent support', 'confidence', 'lift']
].head(10)

In [61]:
display(mejor_reglas_co)

,antecedents,consequents,antecedent support,consequent support,confidence,lift
41,"(FRIJOL CARGAMANTO, CAFÉ, AZÚCAR)",(LECHE),0.05,0.39,0.96,2.44
44,(LECHE),"(FRIJOL CARGAMANTO, CAFÉ, AZÚCAR)",0.39,0.05,0.11,2.44
12,"(CAFÉ, AZÚCAR)",(LECHE),0.32,0.39,0.95,2.41
13,(LECHE),"(CAFÉ, AZÚCAR)",0.39,0.32,0.76,2.41
4,"(FRIJOL CARGAMANTO, ACEITE DE GIRASOL)",(ARROZ),0.30,0.38,0.91,2.41
9,(ARROZ),"(FRIJOL CARGAMANTO, ACEITE DE GIRASOL)",0.38,0.30,0.73,2.41
59,"(PAN TAJADO, CAFÉ, AZÚCAR)",(LECHE),0.03,0.39,0.94,2.39
66,(LECHE),"(PAN TAJADO, CAFÉ, AZÚCAR)",0.39,0.03,0.07,2.39
50,"(HUEVOS, CAFÉ, AZÚCAR)",(LECHE),0.04,0.39,0.92,2.36
55,(LECHE),"(HUEVOS, CAFÉ, AZÚCAR)",0.39,0.04,0.09,2.36


In [63]:
# Filtrar la cesta solo para México
cesta_mx = preparar_cesta(df_limpio, 'México')

# Generar itemsets y reglas de México
frequent_itemsets_mx = apriori(cesta_mx, min_support=0.05, use_colnames=True)
rules_mx = association_rules(frequent_itemsets_mx, metric="lift", min_threshold=1)

# Esta es la tabla que debes mirar para las respuestas 3.3 y 3.4
resultado = rules_mx.sort_values(['lift', 'confidence'], ascending=False)[
    ['antecedents', 'consequents', 'confidence', 'lift']
].head(10)

display(resultado)


,antecedents,consequents,confidence,lift
60,"(CHILE JALAPEÑO, CEBOLLA)","(CILANTRO, TOMATE)",0.92,2.90
57,"(CILANTRO, TOMATE)","(CHILE JALAPEÑO, CEBOLLA)",0.93,2.90
58,"(CILANTRO, CEBOLLA)","(CHILE JALAPEÑO, TOMATE)",0.95,2.88
59,"(CHILE JALAPEÑO, TOMATE)","(CILANTRO, CEBOLLA)",0.90,2.88
23,"(AGUACATE, LIMÓN)",(TOTOPOS),0.93,2.82
26,(TOTOPOS),"(AGUACATE, LIMÓN)",0.75,2.82
56,"(CILANTRO, CHILE JALAPEÑO)","(TOMATE, CEBOLLA)",0.92,2.81
61,"(TOMATE, CEBOLLA)","(CILANTRO, CHILE JALAPEÑO)",0.91,2.81
22,"(AGUACATE, TOTOPOS)",(LIMÓN),0.95,2.69
27,(LIMÓN),"(AGUACATE, TOTOPOS)",0.70,2.69


3.3 Observa las 3 reglas con el Lift más alto para México (1, 3 y 5). **Interprétalas:** ¿Qué te dicen estas asociaciones? ¿Qué tipo de productos son?
Las reglas con mayor Lift (filas 60, 57 y 58) revelan un comportamiento de compra por conveniencia de receta. La asociación entre Chile Jalapeño, Cebolla, Cilantro y Tomate indica que el consumidor mexicano como su proveedor principal para ingredientes de frescos y perecederos.

3.4 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift
Confianza (Confidence): En las reglas 60, 57 y 58, la confianza supera el 0.90 (90%). Esto es estadísticamente muy alto; nos dice que el margen de error al predecir la compra de Cilantro/Tomate si ya se tiene Chile/Cebolla es de apenas el 8% o 10%. Es un flujo de inventario sumamente predecible.
Lift: El valor de 2.90 es el indicador de fuerza. Significa que la probabilidad de que un cliente compre estos cuatro vegetales juntos es 2.9 veces mayor a la probabilidad de que los compre por separado. Esto descarta que la asociación sea fruto del azar o de que los productos sean baratos; es una asociación estratégica de consumo.



3.5 **Recomendación de Negocio:** Basado en estas reglas, ¿qué promoción o estrategia de venta específica podrías sugerir para el mercado mexicano?
DEn lugar de bajar el precio de los productos por separado, Global Retail Inc. puede ofrecer un descuento automático en los Totopos únicamente cuando se detecte la compra simultánea de Aguacates y Limones. Esta estrategia aprovecha que la compra de los vegetales es la que se'dispara' la necesidad del snack. Al incentivar la compra del tercer elemento con un descuento pequeño, se asegura un incremento en el ticket promedio

3.6 Análisis para Colombia

In [ ]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de Colombia. Almacena el resultado en la variable cesta_co.
### TU CÓDIGO AQUÍ ###
cesta_co = preparar_cesta(df_limpio, 'Colombia')

In [ ]:
# TAREA: Aplica el algoritmo apriori con un soporte mínimo del 2%.
# Almacena el resultado en la variable frequent_itemsets_co.
# Muestra los 10 itemsets con el soporte más alto.
### TU CÓDIGO AQUÍ ###
frequent_itemsets_co = apriori(cesta_co, min_support=0.02, use_colnames=True)
frequent_itemsets_co.sort_values('support', ascending=False).head(10)


,support,itemsets
3,0.41,(CAFÉ)
4,0.41,(FRIJOL CARGAMANTO)
0,0.40,(ACEITE DE GIRASOL)
2,0.40,(AZÚCAR)
7,0.39,(LECHE)
1,0.38,(ARROZ)
9,0.35,(QUESO MUZZARELLA)
5,0.34,(HARINA DE MAÍZ)
37,0.32,"(LECHE, CAFÉ)"
31,0.32,"(LECHE, AZÚCAR)"


In [ ]:
# TAREA: Genera las reglas de asociación con un Lift mayor a 2. Almacena el resultado en la variable rules_co.
### TU CÓDIGO AQUÍ ###
rules_co = association_rules(frequent_itemsets_co, metric='lift', min_threshold=2)

In [ ]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
### TU CÓDIGO AQUÍ ###
mejor_reglas_co = rules_co.sort_values(['lift', 'confidence'], ascending=False)[
    ['antecedents', 'consequents', 'antecedent support',
     'consequent support', 'confidence', 'lift']
].head(10)

In [47]:
df_limpio.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536000,94537,HARINA DE MAÍZ,5,2023-01-07 01:09:00,2.76,"17,452.00",Colombia
1,536000,87297,QUESO MUZZARELLA,2,2023-01-07 01:09:00,4.69,"17,779.00",Colombia
2,536001,94537,HARINA DE MAÍZ,4,2023-01-07 11:51:00,2.76,"14,933.00",Colombia
3,536001,87297,QUESO MUZZARELLA,3,2023-01-07 11:51:00,4.69,"14,957.00",Colombia
4,536002,26907,CAFÉ,4,2023-01-02 01:54:00,2.36,"15,202.00",Colombia


In [56]:
mejor_reglas_co = rules_co.sort_values(['lift', 'confidence'], ascending=False)[
    ['antecedents', 'consequents', 'antecedent support',
     'consequent support', 'confidence', 'lift']
].head(10)

In [57]:
display(mejor_reglas_co)

,antecedents,consequents,antecedent support,consequent support,confidence,lift
41,"(FRIJOL CARGAMANTO, CAFÉ, AZÚCAR)",(LECHE),0.05,0.39,0.96,2.44
44,(LECHE),"(FRIJOL CARGAMANTO, CAFÉ, AZÚCAR)",0.39,0.05,0.11,2.44
12,"(CAFÉ, AZÚCAR)",(LECHE),0.32,0.39,0.95,2.41
13,(LECHE),"(CAFÉ, AZÚCAR)",0.39,0.32,0.76,2.41
4,"(FRIJOL CARGAMANTO, ACEITE DE GIRASOL)",(ARROZ),0.30,0.38,0.91,2.41
9,(ARROZ),"(FRIJOL CARGAMANTO, ACEITE DE GIRASOL)",0.38,0.30,0.73,2.41
59,"(PAN TAJADO, CAFÉ, AZÚCAR)",(LECHE),0.03,0.39,0.94,2.39
66,(LECHE),"(PAN TAJADO, CAFÉ, AZÚCAR)",0.39,0.03,0.07,2.39
50,"(HUEVOS, CAFÉ, AZÚCAR)",(LECHE),0.04,0.39,0.92,2.36
55,(LECHE),"(HUEVOS, CAFÉ, AZÚCAR)",0.39,0.04,0.09,2.36


3.7 Observa las 3 reglas con el Lift más alto para Colombia (1, 3 y 5). **Interprétalas:** ¿Qué patrones de consumo específicos del mercado colombiano revelan estas reglas? ¿Son diferentes a las de México?

En Colombia, se observa un patrón principal es la canasta básica familiar completa. El Lift de 2.44 en la regla 41 muestra que quienes compran la base del almuerzo (frijoles) y para el desayuno (café y azúcar), tienen más probabilidad de que se compren estos productos.

3.8 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift
**El soporte** del consecuente (Leche) es de 0.39, lo que significa que la leche está presente en el 39% de todas las transacciones en Colombia. Es un producto esencial en la canata familiar.

**Confianza:** La confianza de 0.96, Indica que el 96% de las veces que alguien lleva la combinación de frijol, café y azúcar, también lleva la leche al carrito.
**Lift:** El Lift de 2.44 confirma que llevar esos tres productos hace que sea más del doble de probable que el cliente compre leche, comparado con un cliente que compra al azar.


3.9 **Recomendación de Negocio:** ¿Qué campaña de marketing (diferente a la de México) podrías diseñar para los clientes colombianos?
Podría diseñar una campaña de un mercado en el cual tebga productos esenciales de la canasta familiar. Dado que la Leche tiene una confianza del 96% con otros productos básicos, lo que también se puede proponer que se haga un combo en el cual esten productos que la gente compra usualamente para que le salga mejor y quiera seguir comprando estos proudctos.